# TC-LoRA on cell states

Baseline 1 of 3. Trains a base diffusion model, freezes it, trains a TC-LoRA
hypernetwork on top, samples from both.

**Paper:** Cho, Ohana, Jacobsen, Jothi, Chen, Mao, Can.
*TC-LoRA: Temporally Modulated Conditional LoRA for Adaptive Diffusion Control.*
NeurIPS 2025 SpaVLE workshop. [arXiv:2510.09561](https://arxiv.org/abs/2510.09561)

**The authors released no code** (checked 2026-09-28). `src/guidance/baselines/tc_lora.py`
is our reimplementation from Eq. 1 and Appendices A-B; its header lists every deviation.
Report this as a reimplementation, not as their result reproduced.

Run top to bottom. On Colab set **Runtime > Change runtime type > GPU** first.

In [ ]:
# Colab bootstrap. Run once.
import importlib.util, subprocess, sys, os

for pkg in ("transformers", "datasets"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

if importlib.util.find_spec("guidance") is None:
    if not os.path.exists("discrete-generative-models-proj-1"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/imperorrp/discrete-generative-models-proj-1.git"],
                       check=True)
    sys.path.insert(0, "discrete-generative-models-proj-1/src")

import torch
from guidance.baselines.tc_lora import TCLoRA
print("device:", "cuda" if torch.cuda.is_available()
      else "CPU  <-- Runtime > Change runtime type > GPU")

## Config

In [ ]:
import math, time, json
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class Cfg:
    seed: int = 0
    batch_size: int = 128
    base_epochs: int = 40
    lora_epochs: int = 20
    lr_base: float = 2e-4
    lr_hyper: float = 1e-4
    K: int = 1000                  # diffusion training steps
    sample_steps: int = 50         # DDIM steps
    rank: int = 4
    hyper_width: int = 256
    limit: int = 0                 # >0 truncates the training set, for smoke tests

cfg = Cfg()
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
print(DEV, asdict(cfg))

## Data

**Replace `make_cell_loaders` with the real Tahoe-100M loader.** The contract:

`(train_loader, val_loader, meta)`, loaders yielding `(x, cond_id)` with `x` of shape
`(B, D)` standardised using train-split statistics only and `cond_id` a long tensor in
`[0, n_clusters)`. `meta` carries `n_conditions`, `shape`, and `cond_text`.

Until then this generates a synthetic mixture so the notebook runs end to end.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset, Subset

def make_cell_loaders(cfg):
    '''REPLACE THIS BODY with the real loader.

    cond_text should describe the biology, not just name the cluster, e.g.
      "Trametinib, a MEK1/2 inhibitor that blocks MAPK/ERK signalling"
    Write these once from a public source BEFORE looking at any results.
    '''
    D, k, n = 50, 3, 6000
    rng = np.random.default_rng(cfg.seed)
    centers = rng.normal(size=(k, D)) * 3.0
    spreads = np.linspace(0.4, 1.4, k)          # clusters differ in spread on purpose
    cond = rng.integers(0, k, size=n)
    x = centers[cond] + rng.normal(size=(n, D)) * spreads[cond][:, None]
    x = (x - x.mean(0)) / (x.std(0) + 1e-8)

    ds = TensorDataset(torch.tensor(x, dtype=torch.float32),
                       torch.tensor(cond, dtype=torch.long))
    cut = int(0.9 * n)
    meta = {"n_conditions": k, "shape": (D,), "SYNTHETIC": True,
            "cond_text": [f"synthetic perturbation cluster {i}" for i in range(k)]}
    return (DataLoader(Subset(ds, range(cut)), cfg.batch_size, shuffle=True, drop_last=True),
            DataLoader(Subset(ds, range(cut, n)), cfg.batch_size, shuffle=False), meta)


train_loader, val_loader, meta = make_cell_loaders(cfg)
if meta.get("SYNTHETIC"):
    print("WARNING: synthetic data. Replace make_cell_loaders before reporting anything.")

xb, cb = next(iter(train_loader))
print("x", tuple(xb.shape), "| cond", tuple(cb.shape), "| conditions", meta["n_conditions"])

## Frozen condition encoder

`h(c)` is a frozen CLIP text embedding of the condition description. Falls back to fixed
random vectors if CLIP will not load, so the notebook still runs — but a fallback run is
**not a result**, and the manifest records it.

In [ ]:
def build_cond_embeddings(texts, device):
    try:
        from transformers import CLIPTokenizer, CLIPTextModel
        name = "openai/clip-vit-base-patch32"
        tok = CLIPTokenizer.from_pretrained(name)
        enc = CLIPTextModel.from_pretrained(name).to(device).eval()
        for p in enc.parameters():
            p.requires_grad_(False)
        out = []
        with torch.no_grad():
            for i in range(0, len(texts), 64):
                b = tok(texts[i:i + 64], padding=True, truncation=True, return_tensors="pt")
                out.append(enc(**{k: v.to(device) for k, v in b.items()}).pooler_output)
        emb = torch.cat(out)
        return emb / emb.norm(dim=-1, keepdim=True), False
    except Exception as e:
        print(f"CLIP unavailable ({type(e).__name__}); using random fallback embeddings.")
        g = torch.Generator().manual_seed(0)
        emb = torch.randn(len(texts), 512, generator=g)
        return (emb / emb.norm(dim=-1, keepdim=True)).to(device), True

COND_EMB, IS_FALLBACK = build_cond_embeddings(meta["cond_text"], DEV)
COND_DIM = COND_EMB.shape[1]
print("condition embeddings:", tuple(COND_EMB.shape), "| fallback:", IS_FALLBACK)

## Backbone

In [ ]:
def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10_000.0) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([args.cos(), args.sin()], dim=-1)


class CellMLP(nn.Module):
    '''Plain MLP. Width follows BranchSBM's flow network (3 x 1024, SELU).'''

    def __init__(self, d=50, n_classes=3, width=1024, tdim=128):
        super().__init__()
        self.tdim, self.null = tdim, n_classes
        self.t_mlp = nn.Sequential(nn.Linear(tdim, width), nn.SiLU(), nn.Linear(width, width))
        self.label = nn.Embedding(n_classes + 1, width)
        self.inp = nn.Linear(d, width)
        self.body = nn.Sequential(nn.SELU(), nn.Linear(width, width),
                                  nn.SELU(), nn.Linear(width, width), nn.SELU())
        self.out = nn.Linear(width, d)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)

    def forward(self, x, t, cond_ids=None):
        if cond_ids is None:
            cond_ids = torch.full((x.shape[0],), self.null, device=x.device, dtype=torch.long)
        h = self.inp(x) + self.t_mlp(timestep_embedding(t, self.tdim)) + self.label(cond_ids)
        return self.out(self.body(h))


def make_backbone():
    return CellMLP(meta["shape"][0], meta["n_conditions"])

net = make_backbone().to(DEV)
print(f"backbone params: {sum(p.numel() for p in net.parameters()):,}")

## Diffusion — cosine schedule, epsilon prediction, DDIM sampler

In [ ]:
def cosine_alpha_bar(K, s=0.008, device="cpu"):
    steps = torch.arange(K + 1, dtype=torch.float32, device=device) / K
    f = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0])[1:].clamp(1e-5, 0.9999)

ABAR = cosine_alpha_bar(cfg.K, device=DEV)
NULL_ID = meta["n_conditions"]

# At the first DDIM step sqrt(abar) is about 0.003, so x0 = (x - sqrt(1-abar) eps)/sqrt(abar)
# divides by a tiny number and any error in eps is amplified ~300x. Without clamping the
# samples blow up and the variance ratio reads in the thousands.
CLIP_X0 = float(torch.cat([x for x, _ in train_loader]).abs().max())
print("x0 clamp:", round(CLIP_X0, 3))


def add_noise(x0):
    n = x0.shape[0]
    k = torch.randint(0, cfg.K, (n,), device=x0.device)
    eps = torch.randn_like(x0)
    ab = ABAR[k].view(n, *([1] * (x0.dim() - 1)))
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps, k, eps


def drop_labels(y, p_drop=0.1):
    '''Condition dropout creates the null branch that guidance needs later.'''
    keep = torch.rand(y.shape[0], device=y.device) >= p_drop
    return torch.where(keep, y, torch.full_like(y, NULL_ID))


@torch.no_grad()
def ddim_sample(eps_fn, n, shape, steps, clip_x0=None, n_snapshots=0):
    '''eps_fn(x, t) -> predicted noise. Deterministic DDIM.

    n_snapshots > 0 also returns that many evenly spaced intermediate states, for the
    denoising-trajectory figure.
    '''
    clip_x0 = CLIP_X0 if clip_x0 is None else clip_x0
    idx = torch.linspace(cfg.K - 1, 0, steps).long().to(DEV)
    keep = set(torch.linspace(0, steps - 1, max(n_snapshots, 1)).long().tolist())
    x = torch.randn(n, *shape, device=DEV)
    snaps = [x.cpu().clone()] if n_snapshots else []
    for i, k in enumerate(idx):
        t = (k.float() / cfg.K).repeat(n)
        eps = eps_fn(x, t)
        ab = ABAR[k]
        ab_prev = ABAR[idx[i + 1]] if i + 1 < len(idx) else torch.ones_like(ab)
        x0 = ((x - (1 - ab).sqrt() * eps) / ab.sqrt()).clamp(-clip_x0, clip_x0)
        x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * eps
        if n_snapshots and i in keep:
            snaps.append(x.cpu().clone())
    return (x, snaps) if n_snapshots else x

## Train the base model

In [ ]:
def run_epochs(model, params, n_epochs, lr, tag, loss_fn, ckpt):
    '''Trains, tracks validation loss, and keeps the best checkpoint.'''
    opt = torch.optim.AdamW(params, lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(n_epochs, 1))
    hist, best, best_ep = [], float("inf"), 0
    for ep in range(1, n_epochs + 1):
        model.train(); tot = num = 0; t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(DEV), y.to(DEV)
            loss = loss_fn(x, y)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            tot += loss.item() * x.shape[0]; num += x.shape[0]
        sched.step()

        model.eval(); vtot = vnum = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEV), y.to(DEV)
                vtot += loss_fn(x, y).item() * x.shape[0]; vnum += x.shape[0]
        val = vtot / vnum
        is_best = val < best
        if is_best:
            best, best_ep = val, ep
            torch.save({"state": model.state_dict(), "epoch": ep, "val": val}, ckpt)
        hist.append({"epoch": ep, "train": tot / num, "val": val, "best": is_best})
        print(f"{tag} epoch {ep:3d}  train {tot/num:.4f}  val {val:.4f}"
              f"{'  *best' if is_best else ''}  ({time.time()-t0:.0f}s)")
    print(f"{tag}: best val {best:.4f} at epoch {best_ep}, saved to {ckpt}")
    return hist


def base_loss(x, y):
    xk, k, eps = add_noise(x)
    return F.mse_loss(net(xk, k.float() / cfg.K, drop_labels(y)), eps)

base_hist = run_epochs(net, list(net.parameters()), cfg.base_epochs, cfg.lr_base,
                       "base", base_loss, "base.pt")

## Attach TC-LoRA and train the hypernetwork

The backbone is frozen; only the hypernetwork trains. The first assertion is the one that
matters — at initialisation the adapted model must match the base exactly, because the
hypernetwork's B head is zero-initialised (Appendix A). If that fails, the adapter is not
a clean control and the comparison means nothing.

In [ ]:
ckpt = torch.load("base.pt", map_location=DEV)
print(f"loading base from epoch {ckpt['epoch']} (val {ckpt['val']:.4f})")
base_frozen = make_backbone().to(DEV)
base_frozen.load_state_dict(ckpt["state"])
base_frozen.eval()

probe_x, probe_y = xb[:4].to(DEV), cb[:4].to(DEV)
probe_t = torch.rand(4, device=DEV)
with torch.no_grad():
    ref = base_frozen(probe_x, probe_t, probe_y).clone()

tc = TCLoRA(base_frozen, cond_dim=COND_DIM, rank=cfg.rank,
            width=cfg.hyper_width, skip_names=("out",)).to(DEV)

with torch.no_grad():
    got = tc(probe_x, probe_t, COND_EMB[probe_y], cond_ids=probe_y)
assert torch.allclose(got, ref, atol=1e-5), (got - ref).abs().max().item()
print("no-op at init confirmed")

assert all(n.startswith("hyper.") for n, p in tc.named_parameters() if p.requires_grad)
print(f"trainable: {tc.n_trainable():,} params, all in the hypernetwork")
print(f"frozen:    {sum(p.numel() for p in base_frozen.parameters()):,} params")
print(f"adapted layers: {len(tc.adapted)}")

In [ ]:
def tc_loss(x, y):
    xk, k, eps = add_noise(x)
    # The hypernetwork reads h(c); the frozen backbone still gets the label id.
    return F.mse_loss(tc(xk, k.float() / cfg.K, COND_EMB[y], cond_ids=y), eps)

tc_hist = run_epochs(tc, [p for p in tc.parameters() if p.requires_grad],
                     cfg.lora_epochs, cfg.lr_hyper, "tc-lora", tc_loss, "tclora.pt")

# restore the best hypernetwork before sampling
best = torch.load("tclora.pt", map_location=DEV)["state"]
tc.hyper.load_state_dict({k[len("hyper."):]: v for k, v in best.items()
                          if k.startswith("hyper.")})
tc.eval()
print("restored best hypernetwork")

## Training curves

Train and validation loss per epoch, with the selected checkpoint marked. This is the
evidence for the model-selection bullet on slides 1c and 1d -- "explain how the final
model was selected".

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for a, (name, hist) in zip(ax, [("base", base_hist), ("tc-lora", tc_hist)]):
    ep = [h["epoch"] for h in hist]
    a.plot(ep, [h["train"] for h in hist], label="train")
    a.plot(ep, [h["val"] for h in hist], label="val")
    best = min(hist, key=lambda h: h["val"])
    a.axvline(best["epoch"], ls="--", c="grey", lw=1)
    a.scatter([best["epoch"]], [best["val"]], c="red", zorder=5,
              label=f"selected (epoch {best['epoch']})")
    a.set_title(f"{name}: best val {best['val']:.4f}")
    a.set_xlabel("epoch"); a.set_ylabel("noise-prediction MSE"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Sample from both

In [ ]:
N = 256
y_eval = torch.arange(N, device=DEV) % meta["n_conditions"]

base_s = ddim_sample(lambda x, t: base_frozen(x, t, y_eval), N, meta["shape"], cfg.sample_steps)
tc_s = ddim_sample(lambda x, t: tc(x, t, COND_EMB[y_eval], cond_ids=y_eval),
                   N, meta["shape"], cfg.sample_steps)
print("sampled:", tuple(base_s.shape), "| NFE per sample:", cfg.sample_steps)

In [ ]:
import matplotlib.pyplot as plt

real = torch.cat([x for x, _ in val_loader]).numpy()
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), sharex=True, sharey=True)
for a, (name, pts) in zip(ax, [("real (val)", real), ("base", base_s.cpu().numpy()),
                               ("tc-lora", tc_s.cpu().numpy())]):
    a.scatter(pts[:, 0], pts[:, 1], s=8, alpha=0.5)
    a.set_title(name); a.set_xlabel("PC1")
ax[0].set_ylabel("PC2"); plt.tight_layout(); plt.show()

## Manifest

In [ ]:
def mmd_rbf(x, y, bandwidth=None, max_n=500, seed=0):
    rng = np.random.default_rng(seed)
    if len(x) > max_n: x = x[rng.choice(len(x), max_n, replace=False)]
    if len(y) > max_n: y = y[rng.choice(len(y), max_n, replace=False)]
    z = np.vstack([x, y])
    d2 = ((z[:, None] - z[None]) ** 2).sum(-1)
    if bandwidth is None:
        med = np.median(d2[d2 > 0])
        bandwidth = float(np.sqrt(med / 2)) if med > 0 else 1.0
    k = np.exp(-d2 / (2 * bandwidth ** 2))
    n, m = len(x), len(y)
    kxx = (k[:n, :n].sum() - np.trace(k[:n, :n])) / max(n * (n - 1), 1)
    kyy = (k[n:, n:].sum() - np.trace(k[n:, n:])) / max(m * (m - 1), 1)
    return float(kxx + kyy - 2 * k[:n, n:].mean()), bandwidth


def variance_ratio(pred, real):
    return float(np.trace(np.cov(pred, rowvar=False)) / np.trace(np.cov(real, rowvar=False)))


report = {
    "method": "TC-LoRA (reimplementation of arXiv:2510.09561)",
    "modality": "cells",
    "config": asdict(cfg),
    "base_best_val": min(h["val"] for h in base_hist),
    "tclora_best_val": min(h["val"] for h in tc_hist),
    "trainable_hypernetwork": tc.n_trainable(),
    "frozen_backbone": sum(p.numel() for p in base_frozen.parameters()),
    "adapted_layers": len(tc.adapted),
    "nfe": cfg.sample_steps,
    "clip_fallback": IS_FALLBACK,
    "synthetic_data": bool(meta.get("SYNTHETIC", False)),
    "device": str(DEV),
}
report.update(globals().get("FID_SCORES", {}))

real = torch.cat([x for x, _ in val_loader]).numpy()
for name, s in [("base", base_s), ("tclora", tc_s)]:
    arr = s.cpu().numpy()
    m, bw = mmd_rbf(arr, real)
    report[f"{name}_mmd"] = m
    report[f"{name}_mmd_bandwidth"] = bw
    report[f"{name}_var_ratio"] = variance_ratio(arr, real)

print(json.dumps(report, indent=2))
with open("manifest_tclora_cells.json", "w") as f:
    json.dump(report, f, indent=2)

## Before reporting any of this

- [ ] `make_cell_loaders` replaced with the real Tahoe-100M loader
- [ ] `cond_text` written from a public source **before** seeing results
- [ ] `clip_fallback: false` in the manifest
- [ ] `base_epochs` and `lora_epochs` at their real values, `limit` at 0
- [ ] Three seeds, mean +/- std
- [ ] Parameter counts and NFE next to every number
- [ ] Described as a **reimplementation** of arXiv:2510.09561, not a reproduction, with the scale gap disclosed: the authors train 251M hypernetwork parameters for 3 days on 8x H100